# Module 1 — Is this a job for AI, and what exactly is the system?

**The problem.** A sponsor says *"we need AI for customer service."* The most expensive mistake an AI architect can make is to start designing. The first job is to decide whether AI is even the right tool, and to turn a vague request into a bounded system with measurable success.

This is a **general tutorial** — it teaches the feasibility framework and the problem-classification decision on illustrative examples. It is not tied to any one project. (The companion `scoping-document` **skill** is what you run on a *real* new project to produce its scoping doc.)

The functions below **derive their verdicts from each project's attributes**, not from its name — so they work on a project you've never seen. The last section proves that with a test.


## 0. The example projects

Four deliberately different asks. Every attribute here is an *input* an architect would establish during scoping; the functions read these fields, never the name.


In [1]:
from dataclasses import dataclass

@dataclass
class Project:
    name: str
    ask: str
    # --- feasibility inputs ---
    data_available: str        # strong | workable | weak | none
    task_specifiable: str      # can you write a correctness rubric?  strong | workable | weak
    error_cost: str            # low | medium | high | catastrophic
    latency_need: str          # batch | interactive | realtime_hard
    hitl_possible: str         # yes | limited | no
    deterministic: bool        # is the core actually a fixed rule?
    regulated_forbidden: bool
    # --- classification inputs (these drive classify(), not the name) ---
    output_type: str           # 'generative' (open-ended) | 'discriminative' (category/score)
    needs_fresh_knowledge: bool # answer needs facts outside the model's training (policies, accounts, case law)
    bounded_labelled: bool      # bounded task with labels available -> classical-ML candidate
    verifiable_citations: bool  # output must cite verifiable sources (legal/medical) -> standard RAG insufficient

PROJECTS = [
    Project("Invoice totals", "Automate adding up invoice line items",
            "strong","strong","high","interactive","yes", deterministic=True, regulated_forbidden=False,
            output_type="discriminative", needs_fresh_knowledge=False, bounded_labelled=False, verifiable_citations=False),
    Project("Support triage", "Route and draft replies for customer support tickets",
            "workable","workable","medium","interactive","yes", deterministic=False, regulated_forbidden=False,
            output_type="generative", needs_fresh_knowledge=True, bounded_labelled=False, verifiable_citations=False),
    Project("Fraud blocking", "Auto-decline transactions the model thinks are fraud, no human",
            "strong","workable","catastrophic","realtime_hard","no", deterministic=False, regulated_forbidden=False,
            output_type="discriminative", needs_fresh_knowledge=False, bounded_labelled=True, verifiable_citations=False),
    Project("Case-law memo", "Draft legal memos citing real case law",
            "weak","weak","high","interactive","limited", deterministic=False, regulated_forbidden=False,
            output_type="generative", needs_fresh_knowledge=True, bounded_labelled=False, verifiable_citations=True),
]
for p in PROJECTS: print(f"- {p.name}: {p.ask}")


- Invoice totals: Automate adding up invoice line items
- Support triage: Route and draft replies for customer support tickets
- Fraud blocking: Auto-decline transactions the model thinks are fraud, no human
- Case-law memo: Draft legal memos citing real case law


## 1. Feasibility — four hard gates, then a dimension review

The **gates** come first and are absolute: if any fires, AI is the wrong tool no matter how good the rest looks. Only if the gates pass do we review the five dimensions — qualitatively. There is no magic score here; a weak dimension doesn't veto the project, it tells you what to strengthen first. (Latency is judged on **P95**, not the median — the tail is what breaks a real workflow.)


In [2]:
def gates(p):
    """Absolute disqualifiers. Any hit => AI is the wrong tool as currently framed."""
    hits = []
    if p.deterministic:            hits.append("core is deterministic — use rules, not a model")
    if p.data_available == "none": hits.append("required data does not exist")
    if p.error_cost == "catastrophic" and p.hitl_possible == "no":
                                   hits.append("zero error tolerance with no human in the loop")
    if p.regulated_forbidden:      hits.append("regulation forbids it")
    return hits

WEAK = {"weak","none","catastrophic"}
def feasibility(p):
    dims = {"data": p.data_available, "task_structure": p.task_specifiable,
            "error_tolerance": p.error_cost, "latency(P95)": p.latency_need,
            "human_in_loop": p.hitl_possible}
    g = gates(p)
    weak_dims = [k for k,v in dims.items() if v in WEAK]
    if g:                 verdict = "DISQUALIFIED"
    elif not weak_dims:   verdict = "fit"
    else:                 verdict = "fit with caveats"
    return {"project": p.name, **dims, "verdict": verdict,
            "strengthen_first": weak_dims, "gates": g}

import pandas as pd
rows = [feasibility(p) for p in PROJECTS]
print(pd.DataFrame(rows)[["project","data","task_structure","error_tolerance",
                          "latency(P95)","human_in_loop","verdict"]].to_string(index=False))
print()
for r in rows:
    if r["gates"]: print(f'{r["project"]}: GATE — ' + "; ".join(r["gates"]))
    elif r["strengthen_first"]: print(f'{r["project"]}: strengthen first -> {r["strengthen_first"]}')


       project     data task_structure error_tolerance  latency(P95) human_in_loop          verdict
Invoice totals   strong         strong            high   interactive           yes     DISQUALIFIED
Support triage workable       workable          medium   interactive           yes              fit
Fraud blocking   strong       workable    catastrophic realtime_hard            no     DISQUALIFIED
 Case-law memo     weak           weak            high   interactive       limited fit with caveats

Invoice totals: GATE — core is deterministic — use rules, not a model
Fraud blocking: GATE — zero error tolerance with no human in the loop
Case-law memo: strengthen first -> ['data', 'task_structure']


**Read it.** *Invoice totals* is gated out — it's arithmetic, a rule, not a model; the strong data and task scores can't buy that back, which is the whole point of a gate. *Fraud blocking* is gated as framed: catastrophic error cost with **no** human in the loop. Reframe it (flag-for-review instead of auto-decline) and the gate clears — finding that reframing is the architect's job. *Support triage* is a clean fit. *Case-law memo* passes the gates but is *fit with caveats*: weak data and task structure are the first things to strengthen, before any build.


## 2. Classify the problem → the simplest approach that can work

Two axes — **generative vs discriminative**, **retrieval vs reasoning** — read from the project's attributes, then the resolution order: prompt → RAG (only if it needs knowledge outside the model) → fine-tune (only after a measured prompting ceiling) → classical ML (bounded, labelled, latency/cost-constrained). A deterministic task short-circuits: it isn't an ML problem at all.


In [3]:
def classify(p):
    # deterministic work is not an ML problem; don't force it onto the axes
    if p.deterministic:
        return {"project": p.name, "axis_1": "n/a", "axis_2": "n/a",
                "recommended": "none — deterministic rules"}
    axis1 = p.output_type
    axis2 = "retrieval" if p.needs_fresh_knowledge else "reasoning"
    # resolution order, derived purely from attributes
    if p.bounded_labelled and p.latency_need == "realtime_hard":
        rec = "classical ML (bounded, labelled, low-latency)"
    elif p.output_type == "discriminative" and p.bounded_labelled:
        rec = "classical ML (discriminative, bounded, labelled)"
    elif p.output_type == "generative" and p.needs_fresh_knowledge and p.verifiable_citations:
        rec = ("RAG — but STANDARD RAG is insufficient here: verifiable citations + high stakes "
               "need citation verification, reranking, and a 'standard-RAG-fails' design (see M2 legal case)")
    elif p.output_type == "generative" and p.needs_fresh_knowledge:
        rec = "RAG (generative + needs knowledge outside training)"
    elif p.output_type == "generative":
        rec = "prompted LLM (try this before anything heavier)"
    else:
        rec = "classifier / classical ML (discriminative, no fresh knowledge needed)"
    return {"project": p.name, "axis_1": axis1, "axis_2": axis2, "recommended": rec}

for r in [classify(p) for p in PROJECTS]:
    print(f'{r["project"]:16} {r["axis_1"]:14} {r["axis_2"]:10} -> {r["recommended"]}')


Invoice totals   n/a            n/a        -> none — deterministic rules
Support triage   generative     retrieval  -> RAG (generative + needs knowledge outside training)
Fraud blocking   discriminative reasoning  -> classical ML (bounded, labelled, low-latency)
Case-law memo    generative     retrieval  -> RAG — but STANDARD RAG is insufficient here: verifiable citations + high stakes need citation verification, reranking, and a 'standard-RAG-fails' design (see M2 legal case)


Note the discipline, now coming from the attributes not a lookup: *Fraud blocking* is discriminative + bounded + latency-tight → **classical ML**, not an LLM, even though 'AI' was the ask. *Support triage* is generative and needs account/policy facts → **RAG**. And *Case-law memo* trips the `verifiable_citations` flag, so it is **not** sent to vanilla RAG — the course's own legal case study is titled *when standard RAG fails*, because citation-grade legal answers need verification and reranking on top. A framework that recommended plain RAG there would be teaching the wrong lesson.


## 3. Decompose the ask — most 'AI systems' are a bundle

A business request is rarely one task. The table below is a **worked illustration** (hand-authored, not computed) of decomposing *"AI for customer service"* — the skill does this derivation live for a real brief. The lesson: the pieces that are deterministic shouldn't touch a model.


In [4]:
decomposition = [
    ("detect language",             "classifier (cheap, bounded)",        "low"),
    ("classify ticket intent",      "classifier / prompted LLM",          "low"),
    ("look up account + entitlements","deterministic DB query",            "n/a — no model"),
    ("retrieve relevant policy",    "RAG retrieval",                      "medium"),
    ("draft the reply",             "prompted LLM, grounded on retrieval", "high"),
    ("decide auto-send vs human",   "business rule + confidence gate",    "n/a — no model"),
]
print(pd.DataFrame(decomposition, columns=["sub-task","approach","criticality"]).to_string(index=False))
print("\n2 of 6 sub-tasks need no model at all — that is the point of decomposing.")


                      sub-task                            approach    criticality
               detect language         classifier (cheap, bounded)            low
        classify ticket intent           classifier / prompted LLM            low
look up account + entitlements              deterministic DB query n/a — no model
      retrieve relevant policy                       RAG retrieval         medium
               draft the reply prompted LLM, grounded on retrieval           high
     decide auto-send vs human     business rule + confidence gate n/a — no model

2 of 6 sub-tasks need no model at all — that is the point of decomposing.


## 4. Meaningful tests — not coverage padding

A test earns its place by asserting a **real decision** the framework must get right, and it carries its reason. We don't test that a constant equals itself. Crucially, the last test runs on a project that is **not** in our list — if the logic were hardcoded per example (an easy trap), that test would fail. It's the test that keeps the framework honest.


In [5]:
import dataclasses

def run_checks():
    checks = []
    def check(name, cond, why):
        checks.append((name, "PASS" if cond else "FAIL", why))

    inv, triage, fraud, legal = PROJECTS

    # a deterministic task must be gated out AND short-circuit classification
    check("deterministic -> gated + no-model",
          feasibility(inv)["verdict"] == "DISQUALIFIED" and classify(inv)["recommended"].startswith("none"),
          "arithmetic must never be sent to a model")

    # the fraud gate is the no-human + catastrophic COMBINATION, not the ML
    check("catastrophic + no HITL -> gated",
          "zero error tolerance with no human in the loop" in gates(fraud),
          "auto-declining with no review is the unsafe part, not the classifier")
    check("...clears when a human reviews",
          gates(dataclasses.replace(fraud, hitl_possible="yes")) == [],
          "reframing to flag-for-review is the architect's fix")

    # bounded + realtime discriminative -> classical ML, not an LLM
    check("bounded+realtime -> classical ML",
          "classical ML" in classify(fraud)["recommended"],
          "an LLM is the wrong tool for a tight-latency bounded classifier")

    # knowledge-needing generation -> RAG
    check("knowledge-needing generation -> RAG",
          classify(triage)["recommended"].startswith("RAG (generative"),
          "grounding beats hoping the base model knows your policies")

    # verifiable-citation task must NOT get vanilla RAG (the legal correction)
    check("verifiable citations -> standard RAG flagged insufficient",
          "STANDARD RAG is insufficient" in classify(legal)["recommended"],
          "citation-grade legal/medical answers need more than naive RAG")

    # GENERALIZATION: a project NOT in our list must classify correctly from its attributes.
    # If classify() were a per-name lookup, this fails — which is exactly why it's here.
    novel = Project("Sentiment tags", "Tag product reviews positive/negative/neutral",
                    "strong","strong","low","interactive","yes", deterministic=False, regulated_forbidden=False,
                    output_type="discriminative", needs_fresh_knowledge=False, bounded_labelled=True, verifiable_citations=False)
    check("generalizes to an unseen project",
          "classical ML" in classify(novel)["recommended"] and feasibility(novel)["verdict"] == "fit",
          "the framework must work on a project it has never seen, not just the four seeded ones")

    df = pd.DataFrame(checks, columns=["check","result","why it matters"])
    print(df.to_string(index=False))
    assert all(r == "PASS" for _,r,_ in checks), "a feasibility/classification invariant broke"
    print("\nall", len(checks), "meaningful checks passed")

run_checks()


                                                    check result                                                                        why it matters
                        deterministic -> gated + no-model   PASS                                              arithmetic must never be sent to a model
                          catastrophic + no HITL -> gated   PASS                  auto-declining with no review is the unsafe part, not the classifier
                           ...clears when a human reviews   PASS                                   reframing to flag-for-review is the architect's fix
                         bounded+realtime -> classical ML   PASS                       an LLM is the wrong tool for a tight-latency bounded classifier
                      knowledge-needing generation -> RAG   PASS                             grounding beats hoping the base model knows your policies
verifiable citations -> standard RAG flagged insufficient   PASS                         citat

Seven tests, seven real decisions the framework must make — each with the reason it exists, and the last one proving the logic generalizes past the examples. That is the bar every skill in this factory holds generated code to: a test asserts a behaviour, contract, or safety property and says why; if it only moves a coverage number, it isn't written.


## Check your understanding

**Q1.** *Invoice totals* rated *strong* on data and task structure but was still rejected. Why can't strong dimensions save it?
<details><summary>answer</summary>The four disqualifiers are gates, not scores. A deterministic core (arithmetic) fails outright — a model only adds cost, latency and error to something a rule does exactly. Strong scores elsewhere can't buy back a failed gate; that's why gates run first and separately from the dimension review.</details>

**Q2.** *Fraud blocking* was gated out, then cleared by one change. What was it, and what does it teach about the architect's role?
<details><summary>answer</summary>Adding a human in the loop (flag-for-review instead of auto-decline). The ML was never the unsafe part — the *autonomy* was. The architect's job is to find the reframing that keeps the value and removes the gate.</details>

**Q3.** *Support triage* and *Case-law memo* both need fresh knowledge, so both are 'retrieval'. Why does only one get plain RAG?
<details><summary>answer</summary>Case-law carries the `verifiable_citations` flag: a legal memo must cite real, checkable cases, and the cost of a fabricated citation is high. Vanilla RAG can still surface plausible-but-wrong passages, so it needs citation verification and reranking on top — the course's 'when standard RAG fails' case. Triage has no such flag, so plain RAG is proportionate.</details>

**Q4.** The decomposition sent 2 of 6 sub-tasks to 'no model'. Why is finding those the point?
<details><summary>answer</summary>Every sub-task needlessly routed through a model adds cost, latency and a new failure mode. Account lookup and the auto-send decision are deterministic; keeping them deterministic shrinks the blast radius and the bill, and sharpens what the model is actually for.</details>

**Q5.** Why does the test suite include a project that isn't in the examples list?
<details><summary>answer</summary>To prove the logic is derived from attributes, not hardcoded per example. A per-name lookup table would pass every test keyed on the four known projects and silently fail on the fifth. The generalization test is the one that catches that class of bug.</details>

**Q6.** Why is latency judged on P95 rather than the median?
<details><summary>answer</summary>LLM latency has a long tail. A good median hides the P95/P99 requests that blow the workflow's time budget — and those stack across sequential AI calls. You design and alert on the tail, not the comfortable middle.</details>
